# QuestionnaireResponse → US Core Observations

Transforms a FHIR `QuestionnaireResponse` (and its paired `Questionnaire`) into a set of US Core `Observation` resources:

- one **panel** Observation per group, linked to its members via `hasMember`
- one **item** Observation per answered question
- `multiselect-item` Observations when a single question has multiple `answer` entries

All resources are handled as plain Python dicts (no FHIR class library); configuration is loaded from an external YAML file so the same logic can be re-targeted at any survey. See `.claude/scripts/qr_obs_transform.config.yml` in the USCDIV7 repo for an example.

## Imports

In [ ]:
import copy
import json
from pathlib import Path
import yaml

## Configuration

All run-specific settings live in an external YAML config so the notebook stays generic. Point `CONFIG_PATH` at the config for the survey you want to transform.

In [ ]:
CONFIG_PATH = Path('/Users/ehaas/Documents/FHIR/USCDIV7/.claude/scripts/qr_obs_transform.config.yml')

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

profile              = cfg['profile']
category             = cfg['category']
category_path        = Path(cfg['category_path'])
survey_name          = cfg['survey_name']
survey_code          = cfg['survey_code']
survey_code_name     = cfg['survey_code_name']
create_survey_panel  = cfg.get('create_survey_panel', True)
template_path        = Path(cfg['template_path'])
qr_path              = Path(cfg['qr_path'])
q_path               = Path(cfg['q_path'])
out_dir              = Path(cfg['out_dir'])

out_dir.mkdir(parents=True, exist_ok=True)
out_dir

## I/O helpers

`load_yaml` / `dump_yaml` wrap PyYAML with formatting that matches the rest of the IG's example YAML files.

In [ ]:
def load_yaml(path):
    with open(path) as f:
        return yaml.safe_load(f)

def dump_yaml(obj, path):
    with open(path, 'w') as f:
        yaml.safe_dump(obj, f, default_flow_style=False, sort_keys=False, width=120)

## Load source resources

`q_obj` (Questionnaire) supplies item metadata (units, help text). `qr_obj` (QuestionnaireResponse) supplies answers. `template` is the skeleton Observation reused for every generated resource.

In [ ]:
qr_obj   = load_yaml(qr_path)
q_obj    = load_yaml(q_path)
template = load_yaml(template_path)

qr_obj['id'], q_obj.get('title')

## Category lookup

Looks up the configured `category` code in the US Core category CodeSystem. Returns `None` if not present (so the template's `survey` category stays as the only category).

In [ ]:
def get_category(code):
    if category_path.suffix in {'.yml', '.yaml'}:
        cs = load_yaml(category_path)
    else:
        with open(category_path) as f:
            cs = json.load(f)
    for cat in cs.get('concept', []):
        if cat.get('code') == code:
            return {'coding': [{'system': cs['url'], 'code': cat['code'], 'display': cat['display']}]}
    return None

get_category(category)

## Write Observation to file

In [ ]:
def write_out(obs):
    out_path = out_dir / f'Observation-{obs["id"]}.yml'
    dump_yaml(obs, out_path)

## Update panel members

After a group's child Observations are written, re-open the parent panel YAML and populate its `hasMember` array with references to the child Observation ids.

In [ ]:
def update_members(parent, members):
    if isinstance(parent, str):
        parent = parent.replace('/', '')
    in_path = out_dir / f'Observation-{survey_name}-panel-example-{parent}.yml'
    obs = load_yaml(in_path)
    obs['hasMember'] = [{'reference': f'Observation/{m}'} for m in members]
    write_out(obs)

## Questionnaire lookups

`get_units` reads the `questionnaire-unit` extension on the matching Questionnaire item (drives `valueQuantity.unit`/`code`).

`get_help_text` returns the help-text `display` child item (used as the Observation `note`).

In [ ]:
def get_units(link_id):
    target = f'/{link_id}'
    for item in q_obj.get('item', []):
        if item.get('linkId') == target:
            for ext in item.get('extension', []):
                if ext.get('url') == 'http://hl7.org/fhir/StructureDefinition/questionnaire-unit':
                    return ext.get('valueCoding', {}).get('code')
    return None

def get_help_text(link_id):
    target = f'/{link_id}'
    for item in q_obj.get('item', []):
        if item.get('linkId') == target:
            for child in item.get('item', []) or []:
                if child.get('type') == 'display':
                    return child.get('text')
    return None

## Value builders

Translate a `QuestionnaireResponse.item.answer` entry into the appropriate `Observation.value[x]`:
- `valueCoding` → `valueCodeableConcept`
- `valueString` → `valueString`
- `valueDecimal` → `valueQuantity` (UCUM units from the Questionnaire item)

In [ ]:
def make_ccode(coding):
    return {'coding': [coding], 'text': coding.get('display')}

def make_quantity(value, units):
    if not units:
        return {'value': value}
    unit_label = units.replace('{', '').replace('}', '').replace('/a', ' per year')
    return {
        'value': value,
        'unit': unit_label,
        'system': 'http://unitsofmeasure.org',
        'code': units,
    }

def apply_answer(obs, answer):
    if 'valueCoding' in answer:
        obs['valueCodeableConcept'] = make_ccode(answer['valueCoding'])
    elif 'valueString' in answer:
        obs['valueString'] = answer['valueString']
    elif 'valueDecimal' in answer:
        units = get_units(obs['code']['coding'][0]['code'])
        obs['valueQuantity'] = make_quantity(answer['valueDecimal'], units)
    return obs

## Observation factory

Deep-copies the template and fills in id, meta extensions (instance-name / instance-description), profile, category, code, subject, performer, derivedFrom, and the value or hasMember placeholder.

In [ ]:
def create_obs(obs_id, text, obs_type='item', answer=None):
    obs = copy.deepcopy(template)
    obs['id'] = f'{survey_name}-{obs_type}-example-{obs_id}'
    obs['meta']['extension'][0]['valueString'] = f'{survey_name} {obs_type} Example {obs_id}'.title()
    obs['meta']['extension'][1]['valueMarkdown'] = (
        f'This is a {obs["meta"]["extension"][0]["valueString"]} ({text}) '
        f'for the *US Core Observation Screening Assessment Profile*'
    )
    obs['meta']['profile'] = [profile]
    extra_cat = get_category(category)
    if extra_cat:
        obs['category'].append(extra_cat)
    obs['code']['coding'][0]['code'] = obs_id
    obs['code']['coding'][0]['display'] = text
    obs['code']['text'] = text
    if answer:
        apply_answer(obs, answer)
        obs.pop('hasMember', None)
    else:
        obs['hasMember'] = []
    obs['effectiveDateTime'] = qr_obj.get('authored')
    obs['subject']['reference'] = qr_obj['subject']['reference']
    author_ref = qr_obj.get('author', {}).get('reference')
    if author_ref:
        obs.setdefault('performer', [{}])[0]['reference'] = author_ref
    obs['derivedFrom'] = [{'reference': f'QuestionnaireResponse/{qr_obj["id"]}'}]
    qr_meta_ext = qr_obj.get('meta', {}).get('extension', [])
    if qr_meta_ext and 'valueString' in qr_meta_ext[0]:
        obs['derivedFrom'][0]['display'] = qr_meta_ext[0]['valueString']
    help_text = get_help_text(obs['code']['coding'][0]['code'])
    if help_text:
        obs['note'] = [{'text': help_text}]
    return obs

## Recursive walker

`walk_items` descends the nested `QuestionnaireResponse.item` tree:

1. **Group (no answer, has nested items)** → write a panel Observation, recurse.
2. **Unanswered leaf** → skip.
3. **Single answer** → write an item Observation.
4. **Multiple answers** → write a `multiselect-item` Observation per answer.

After processing all children of a parent, the panel's `hasMember` array is updated. If the QR has no enclosing group and `create_survey_panel` is true, a synthetic panel is created from `survey_code` / `survey_code_name`.

In [ ]:
def walk_items(container, parent=None):
    children = container.get('item')
    if not children:
        return
    member_ids = []
    for child in children:
        link_id = child.get('linkId', '')
        code = link_id.split('/')[-1]
        text = child.get('text', '')
        answers = child.get('answer')
        if not answers:
            if not child.get('item'):
                continue
            panel = create_obs(code, text, 'panel')
            member_ids.append(panel['id'])
            write_out(panel)
        elif len(answers) > 1:
            for idx, ans in enumerate(answers):
                multi_code = f'{code}-answer{idx}'
                item = create_obs(multi_code, text, 'multiselect-item', ans)
                member_ids.append(item['id'])
                write_out(item)
        else:
            item = create_obs(code, text, 'item', answers[0])
            member_ids.append(item['id'])
            write_out(item)
        walk_items(child, link_id)
    if parent is not None:
        try:
            update_members(parent, member_ids)
            return
        except FileNotFoundError:
            pass
    if create_survey_panel:
        survey_panel = create_obs(survey_code, survey_code_name, 'panel')
        write_out(survey_panel)
        update_members(survey_code, member_ids)

walk_items(qr_obj)